In [1]:
!pip install timm --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 86.9 MB/s eta 0:00:00:00:0100:01


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/skin-cancer-mnist-ham10000


In [3]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import timm  # Pretrained models


In [9]:
df = pd.read_csv('/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv')

In [10]:
# Combine image folders into a single directory
all_image_dir = '/kaggle/working/all_images/'

import shutil
import os

# Create the new directory if it doesn't exist
os.makedirs(all_image_dir, exist_ok=True)

# Source directories
dirs_to_merge = [
    '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1/',
    '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2/',
]

# Copy all images into one directory
for dir_path in dirs_to_merge:
    for fname in os.listdir(dir_path):
        src = os.path.join(dir_path, fname)
        dst = os.path.join(all_image_dir, fname)
        if not os.path.exists(dst):
            shutil.copy(src, dst)


In [11]:
df['path'] = df['image_id'].apply(lambda x: os.path.join(all_image_dir, x + '.jpg'))
df = df[df['path'].apply(os.path.exists)]


In [13]:
from torchvision import transforms
from sklearn.model_selection import train_test_split
from PIL import Image
import torch
import os

# Map diagnosis labels to integers
label_map = {label: idx for idx, label in enumerate(df['dx'].unique())}
df['label'] = df['dx'].map(label_map)

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Apply undersampling to balance classes
min_count = df['dx'].value_counts().min()
balanced_df = df.groupby('dx', group_keys=False).apply(
    lambda x: x.sample(min(len(x), min_count), random_state=42)
).reset_index(drop=True)

print("Balanced class distribution:\n", balanced_df['dx'].value_counts())

# Load images and labels
images = []
labels = []

for _, row in balanced_df.iterrows():
    img = Image.open(row['path']).convert('RGB')
    img = transform(img)
    images.append(img)
    labels.append(row['label'])

images = torch.stack(images)
labels = torch.tensor(labels)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, stratify=labels, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


/tmp/ipykernel_35/493660276.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = df.groupby('dx', group_keys=False).apply(


Balanced class distribution:
 dx
akiec    115
bcc      115
bkl      115
df       115
mel      115
nv       115
vasc     115
Name: count, dtype: int64
Train shape: torch.Size([644, 3, 224, 224])
Test shape: torch.Size([161, 3, 224, 224])


In [14]:
class SkinDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        label = self.df.iloc[idx]['label']
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


In [15]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'])
train_ds = SkinDataset(train_df, transform=transform)
val_ds = SkinDataset(val_df, transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)


In [17]:
import timm
import torch

# Detect device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Number of classes in the classification task
num_classes = df['label'].nunique()

# Create the model
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=num_classes)
model.to(device)


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2

In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def train_model(model, train_loader, val_loader, epochs=50):
    for epoch in range(epochs):
        model.train()
        total_loss, correct = 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()


            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, Train Acc: {correct/len(train_loader.dataset):.4f}")


In [19]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
    print(f"Validation Accuracy: {correct/len(loader.dataset):.4f}")


In [20]:
train_model(model, train_loader, val_loader, epochs=50)
evaluate(model, val_loader)


Epoch 1, Loss: 1.0390, Train Acc: 0.6891
Epoch 2, Loss: 0.3552, Train Acc: 0.8736
Epoch 3, Loss: 0.1810, Train Acc: 0.9367
Epoch 4, Loss: 0.0933, Train Acc: 0.9713
Epoch 5, Loss: 0.0552, Train Acc: 0.9855
Epoch 6, Loss: 0.0411, Train Acc: 0.9895
Epoch 7, Loss: 0.0262, Train Acc: 0.9933
Epoch 8, Loss: 0.0280, Train Acc: 0.9923
Epoch 9, Loss: 0.0250, Train Acc: 0.9929
Epoch 10, Loss: 0.0182, Train Acc: 0.9944
Epoch 11, Loss: 0.0228, Train Acc: 0.9933
Epoch 12, Loss: 0.0158, Train Acc: 0.9959
Epoch 13, Loss: 0.0175, Train Acc: 0.9951
Epoch 14, Loss: 0.0140, Train Acc: 0.9968
Epoch 15, Loss: 0.0185, Train Acc: 0.9926
Epoch 16, Loss: 0.0231, Train Acc: 0.9921
Epoch 17, Loss: 0.0330, Train Acc: 0.9891
Epoch 18, Loss: 0.0276, Train Acc: 0.9893
Epoch 19, Loss: 0.0214, Train Acc: 0.9931
Epoch 20, Loss: 0.0138, Train Acc: 0.9961
Epoch 21, Loss: 0.0159, Train Acc: 0.9940
Epoch 22, Loss: 0.0097, Train Acc: 0.9976
Epoch 23, Loss: 0.0097, Train Acc: 0.9966
Epoch 24, Loss: 0.0145, Train Acc: 0.9950
E

In [21]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    predictions = []
    labels_list = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            predictions.extend(predicted.cpu().numpy())
            labels_list.extend(labels.cpu().numpy())
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    print(f"Validation Accuracy: {acc:.4f}")
    return predictions, labels_list


In [22]:
preds, true_labels = evaluate(model, val_loader)


Validation Accuracy: 0.8787


In [27]:
# Save the model
torch.save(model.state_dict(), "efficientnet_skin_disease1.pth")


In [28]:
torch.save(model, "efficientnet_skin_disease_full1.pth")


In [ ]:
def predict_single_image(image_path, model):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)
        predicted_class = le.inverse_transform([predicted.cpu().item()])[0]

    print(f"Predicted class: {predicted_class}")


In [ ]:
predict_single_image("", model)
